In [1]:
import torch
import torch.nn as nn
import requests
import zipfile
import io
from collections import Counter
from torch.utils.data import DataLoader,Dataset
import re
import pandas as pd

In [2]:
url = "http://mattmahoney.net/dc/text8.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
text = z.read("text8").decode("utf-8")
tokens = text.split()
print("Total tokens:", len(tokens))

Total tokens: 17005207


In [3]:
min_freq=5
counter=Counter(tokens)
vocab={}
for word,freq in counter.items():
    if freq>=min_freq:
        vocab[word]=len(vocab)
print(len(vocab))


71290


In [4]:
word2index={word:index for word,index in vocab.items()}
index2word={index:word for word,index in vocab.items()}

In [5]:
pairs=[]
windows_size=2
indexed=[vocab[word] for word in tokens if word in vocab]

In [6]:
for i,center_word in enumerate(indexed):
    for j in range(-windows_size,windows_size+1):
        if (j!=0 and 0<=i+j<len(indexed)):
            contextual_word=indexed[i+j]
            pairs.append((center_word,contextual_word))

In [7]:
len(pairs)

66875370

In [8]:
import torch.nn.functional as F
class SkipgramDataset(Dataset):
    def __init__(self,pairs,vocab_size):
        self.pairs=pairs
        self.vocab_size=vocab_size
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self,index):
        central,contextual=self.pairs[index]
        central_one_hot=F.one_hot(torch.tensor(central),num_classes=self.vocab_size).float()
        return central_one_hot,torch.tensor(contextual)

In [22]:
data=SkipgramDataset(pairs,len(vocab))

In [23]:
class skipgram_model(nn.Module):
    def __init__(self,vocab_size,output_dim):
        super(skipgram_model,self).__init__()
        self.hidden_layer=nn.Linear(vocab_size,output_dim,bias=False)
        self.output_layer=nn.Linear(output_dim,vocab_size,bias=False)
    def forward(self,input):
        x=self.hidden_layer(input)
        output=self.output_layer(x)
        return output
    

In [24]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [25]:
model=skipgram_model(len(vocab),100).to(device)

In [26]:
optimizer=torch.optim.Adam(model.parameters())
loss_fn=nn.CrossEntropyLoss()

In [27]:
loader=DataLoader(data,batch_size=1024,shuffle=True,num_workers=2)

In [ ]:
epochs=10
from tqdm import tqdm
for i  in range(epochs):
    loss=0
    model.train()
    loop = tqdm(loader, desc=f"Epoch {i+1}/{epochs}", leave=False)
    for (x_batch,y_batch) in loop:
        optimizer.zero_grad()
        x_batch,y_batch=x_batch.to(device),y_batch.to(device)
        y_pred=model(x_batch)
        batch_loss=loss_fn(y_pred,y_batch)
        batch_loss.backward()
        optimizer.step()
        loss+=batch_loss.item()
         #Update tqdm live loss
        loop.set_postfix(loss=batch_loss.item())
    print(f'Training Loss over epoch {i+1} is {loss/len(loader)}')

        

Epoch 1/10:   1%|▏         | 850/65308 [01:50<2:03:00,  8.73it/s, loss=7.48]